# Run All OMOP CDM Tables

This notebook orchestrates the creation/refresh of all OMOP CDM Gold layer streaming tables.

## Standalone Architecture

This pipeline is **independent** of the FHIR Bronze/Silver ingestion pipeline. It reads from
Silver tables using Delta Streaming, automatically processing new records as they arrive.

```
Silver Tables (managed by FHIR pipeline)
        ↓ Streaming (automatic CDC)
Gold OMOP Tables (managed by this pipeline)
```

## Execution Order

1. **Foundation Tables** (parallel): `person`, `care_site`, `provider`
2. **Visit Table**: `visit_occurrence` (depends on person)
3. **Clinical Tables** (parallel): `condition_occurrence`, `drug_exposure`, `procedure_occurrence`, `measurement`, `observation`

## Usage

- **Manual Run**: Execute this notebook to refresh all OMOP tables
- **Scheduled Run**: Use the `omop_gold_pipeline` job for automated execution

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';  -- Where FHIR Silver tables live
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';      -- Where OMOP Gold tables live

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);

SELECT 
  catalog_use AS catalog,
  silver_schema AS fhir_silver_schema,
  gold_schema AS omop_gold_schema;

In [ ]:
-- Create Gold schema if not exists
DECLARE OR REPLACE VARIABLE create_schema_stmt STRING;
SET VARIABLE create_schema_stmt = 'CREATE SCHEMA IF NOT EXISTS ' || catalog_use || '.' || gold_schema;
EXECUTE IMMEDIATE create_schema_stmt;

USE IDENTIFIER(catalog_use || '.' || gold_schema);
SELECT current_catalog() AS catalog, current_schema() AS schema;

## Pre-flight Check: Verify Silver Tables Exist

In [ ]:
-- Check which Silver tables are available
DECLARE OR REPLACE VARIABLE check_silver_stmt STRING;

SET VARIABLE check_silver_stmt = "
SELECT 
  table_name,
  CASE 
    WHEN table_name = 'patient' THEN 'person'
    WHEN table_name = 'encounter' THEN 'visit_occurrence'
    WHEN table_name = 'condition' THEN 'condition_occurrence'
    WHEN table_name = 'medicationrequest' THEN 'drug_exposure'
    WHEN table_name = 'procedure' THEN 'procedure_occurrence'
    WHEN table_name = 'observation' THEN 'measurement/observation'
    WHEN table_name = 'practitioner' THEN 'provider'
    WHEN table_name = 'organization' THEN 'care_site'
    ELSE 'not mapped'
  END AS omop_target
FROM " || catalog_use || ".information_schema.tables
WHERE table_schema = '" || silver_schema || "'
ORDER BY table_name
";

EXECUTE IMMEDIATE check_silver_stmt;

## Phase 1: Foundation Tables (Person, Care Site, Provider)

In [ ]:
%run "./01 - Person"

In [ ]:
%run "./08 - Provider and Care Site"

## Phase 2: Visit Occurrence

In [ ]:
%run "./02 - Visit Occurrence"

## Phase 3: Clinical Tables

In [ ]:
%run "./03 - Condition Occurrence"

In [ ]:
%run "./04 - Drug Exposure"

In [ ]:
%run "./05 - Procedure Occurrence"

In [ ]:
%run "./06 - Measurement"

In [ ]:
%run "./07 - Observation"

## Summary: OMOP Table Statistics

In [ ]:
-- Summary of all OMOP tables
SELECT 'person' AS omop_table, COUNT(*) AS row_count FROM person
UNION ALL SELECT 'visit_occurrence', COUNT(*) FROM visit_occurrence
UNION ALL SELECT 'condition_occurrence', COUNT(*) FROM condition_occurrence
UNION ALL SELECT 'drug_exposure', COUNT(*) FROM drug_exposure
UNION ALL SELECT 'procedure_occurrence', COUNT(*) FROM procedure_occurrence
UNION ALL SELECT 'measurement', COUNT(*) FROM measurement
UNION ALL SELECT 'observation', COUNT(*) FROM observation
UNION ALL SELECT 'provider', COUNT(*) FROM provider
UNION ALL SELECT 'care_site', COUNT(*) FROM care_site
ORDER BY omop_table;